# Retrieval-Augmented Generation

Retrieval-augmented generation (RAG) supplies a language model with relevant information retrieved from an external source before it answers a question.

This notebook uses George Orwell’s *1984* as that source. We extract the text, split it into overlapping passages, and retrieve passages relevant to a question. Those passages become context for the model’s answer.

We will compare keyword-based retrieval with BM25 and embedding-based retrieval with FAISS, then inspect how the retrieved context supports the generated response.

## Finding passages with BM25

Before generating an answer, we first need to find relevant information.

BM25 ranks passages using the query’s terms, accounting for how frequently they occur, how common they are across the collection, and passage length.

We begin by extracting text from *1984* and splitting it into passages. We then search those passages and inspect the results directly, without using a language model.

In [1]:
from pathlib import Path
from pypdf import PdfReader

repo_root = Path.cwd()
if repo_root.name == "examples":
    repo_root = repo_root.parent

pdf_path = repo_root / "data" / "orwell1984.pdf"
reader = PdfReader(pdf_path)
text = "\n".join(page.extract_text() or "" for page in reader.pages)

words = text.split()
passages = [
    " ".join(words[i:i + 150])
    for i in range(0, len(words), 150)
]

if not passages:
    raise ValueError("No text passages were extracted from the PDF.")

print(f"Created {len(passages):,} passages of up to 150 words.")
print("\nFirst passage preview:\n")
print(passages[0][:300])

Created 709 passages of up to 150 words.

First passage preview:

19 8 4 nineteen eighty-four a novel George Orwell NINETEEN EIGHTY-FOUR ERIC ARTHUR BLAIR was born in British India in 1903, the son of a Civil Servant working in the Opium Department. He attended Eton, but his family could not pay for university, so in 1922 he joined the Indian Imperial Police. He c


In [2]:
import bm25s

retriever = bm25s.BM25(corpus=passages)
retriever.index(bm25s.tokenize(passages))

query = "Who is Big Brother?"
query_tokens = bm25s.tokenize([query])
results, scores = retriever.retrieve(query_tokens, k=min(3, len(passages)))

print(f"Query: {query}")

for i, (passage, score) in enumerate(zip(results[0], scores[0]), 1):
    print(f"\nPassage {i} | BM25 score: {score:.3f}")
    print(passage)

Query: Who is Big Brother?

Passage 1 | BM25 score: 3.700
back of the neck i dont care down with big brother—— He sat back in his chair, slightly ashamed of himself, and laid down the pen. The next moment he started violently. There was a knocking at the door. Already! He sat as still as a mouse, in the futile hope that who- ever it was might go away after a single attempt. But no, the knocking was repeated. The worst thing of all would be to delay. His heart was thumping like a drum, but his face, from long habit, was probably expressionless. He got up and moved heavily towards the door. 20 Chapter 2 As he put his hand to the door-knob Winston saw that he had left the diary open on the table. DOWN WITH BIG BROTHER was written all over it, in letters almost big enough to be legible across the room. It was an inconceivably

Passage 2 | BM25 score: 3.653
doing it. For a second, two seconds, they had exchanged an equiv- ocal glance, and that was the end of the story. But even that was a m

### About the results

The passages are ranked by their BM25 scores, with the highest-scoring result first. These scores measure relevance based on query terms; they are not probabilities or confidence scores.

Read the passages to check whether they help answer “Who is Big Brother?” A passage may mention the name without explaining who he is.

This is information retrieval: we have selected existing text from the book, but have not generated an answer.

## Retrieving passages with embeddings

BM25 ranks passages by matching query terms. Embedding-based retrieval compares numerical representations of the query and passages, which can help find related content even when the wording differs.

We encode the same passages with `all-MiniLM-L6-v2` and use FAISS to search their vectors. Keeping the passages and question unchanged lets us compare the retrieved results.

In [3]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = embed_model.encode(
    passages,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"Indexed {index.ntotal:,} passages.")
print(f"Embedding dimensions: {embeddings.shape[1]}")

Indexed 709 passages.
Embedding dimensions: 384


In [4]:
# Encode the same question used for BM25 with the passage embedding model.
query_embedding = embed_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

# With normalized vectors, inner-product search gives cosine similarity.
scores, indices = index.search(query_embedding, k=min(3, index.ntotal))

print(f"Query: {query}")

# Show the retrieved text so readers can compare its relevance with BM25.
for i, (passage_index, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"\nPassage {i} | Similarity: {score:.3f}")
    print(passages[passage_index])

Query: Who is Big Brother?

Passage 1 | Similarity: 0.657
all-powerful. Every success, every achieve- ment, every victory, every scientiﬁ c discovery, all knowledge, all wisdom, all happiness, all virtue, are held to issue directly from his leadership and inspiration. Nobody has ever seen Big Brother. He is a face on the hoardings, a voice on the telescreen. We may be reasonably sure that he will never die, and there is already considerable uncertainty as to when he was born. Big Brother is the guise in which the Party chooses to exhibit itself to the world. His function is to act as a focusing point for love, fear, and reverence, emotions which are more easily felt towards an individual than towards an organization. Below Big Brother comes the Inner Party. Its numbers limited to six millions, or something less than 2 per cent of the population of Oceania. Below the Inner Party comes the Outer Party, which, if the Inner

Passage 2 | Similarity: 0.578
to have criminal tendencies. At sev

### Comparing the retrieval results

Both methods searched the same passages using the same question. Compare whether their top results explain who Big Brother is or simply mention him.

BM25 scores reflect term matches, while the FAISS scores here are cosine similarities between embeddings. Their numerical values are not directly comparable.

Embedding-based retrieval may find related wording that BM25 misses, but it is not automatically more relevant. Inspecting the passages helps us judge which results provide useful evidence for an answer.

## Generating an answer from retrieved passages

We now give a language model the passages retrieved by the embedding search and ask it to answer the same question.

This completes the RAG workflow: retrieve relevant text, include it in the prompt, and generate an answer grounded in that context. We label the passages so the model can cite its sources and ask it to say when the provided text is insufficient.

In [5]:
import torch
from IPython.display import Markdown, display
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).eval()

# Keep the original passage IDs from the FAISS results.
context = "\n\n".join(
    f"[Passage {idx}] {passages[idx]}"
    for idx in indices[0]
)

messages = [
    {
        "role": "system",
        "content": (
            "Answer using only the passages provided. "
            "Cite their IDs, such as [Passage 12]. "
            "If the answer is missing, say so."
        ),
    },
    {"role": "user", "content": f"Question: {query}\n\n{context}"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False)

answer = tokenizer.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
).strip()

display(Markdown("## Model response"))
display(Markdown(answer))

display(Markdown("## Retrieved passages — George Orwell, *1984*"))
for idx in indices[0]:
    display(Markdown(f"### Passage {idx} — `passages[{idx}]`"))
    print(passages[idx])

all-powerful. Every success, every achieve- ment, every victory, every scientiﬁ c discovery, all knowledge, all wisdom, all happiness, all virtue, are held to issue directly from his leadership and inspiration. Nobody has ever seen Big Brother. He is a face on the hoardings, a voice on the telescreen. We may be reasonably sure that he will never die, and there is already considerable uncertainty as to when he was born. Big Brother is the guise in which the Party chooses to exhibit itself to the world. His function is to act as a focusing point for love, fear, and reverence, emotions which are more easily felt towards an individual than towards an organization. Below Big Brother comes the Inner Party. Its numbers limited to six millions, or something less than 2 per cent of the population of Oceania. Below the Inner Party comes the Outer Party, which, if the Inner
to have criminal tendencies. At seventeen he had been a district organizer of the Junior Anti- Sex League. At nineteen he ha

## Model response

Big Brother is the face on the hoardings, a voice on the telescreen, and the Inner Party is the number six million, or something less than 2% of the population of Oceania.

## Retrieved passages — George Orwell, *1984*

### Passage 474 — `passages[474]`

### Passage 107 — `passages[107]`

### Passage 38 — `passages[38]`

### About the output

The model’s response is generated from the passages returned by FAISS. Citations such as `[Passage 12]` refer to the original passage IDs, and `passages[12]` locates that text in our collection.

The retrieved passages appear below the answer so we can check whether they support its claims. These IDs identify chunks of extracted text, not PDF page numbers.

RAG supplies relevant context, but the model can still omit citations or make unsupported claims. Compare its answer with the retrieved text rather than treating a citation as proof.